<a href="https://colab.research.google.com/github/ixrlts/data-analytics-case-studies/blob/main/projects/taxis/02_tip_rate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = sns.load_dataset('taxis')
df.head(10)

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan
5,2019-03-11 10:37:23,2019-03-11 10:47:31,1,0.49,7.5,2.16,0.0,12.96,yellow,credit card,Times Sq/Theatre District,Midtown East,Manhattan,Manhattan
6,2019-03-26 21:07:31,2019-03-26 21:17:29,1,3.65,13.0,2.00,0.0,18.80,yellow,credit card,Battery Park City,Two Bridges/Seward Park,Manhattan,Manhattan
7,2019-03-22 12:47:13,2019-03-22 12:58:17,0,1.40,8.5,0.00,0.0,11.80,yellow,NaN,Murray Hill,Flatiron,Manhattan,Manhattan
8,2019-03-23 11:48:50,2019-03-23 12:06:14,1,3.63,15.0,1.00,0.0,19.30,yellow,credit card,East Harlem South,Midtown Center,Manhattan,Manhattan
9,2019-03-08 16:18:37,2019-03-08 16:26:57,1,1.52,8.0,1.00,0.0,13.30,yellow,credit card,Lincoln Square East,Central Park,Manhattan,Manhattan


In [21]:
df['tip_pct'] = df['tip'] / df['fare'] * 100
card = df[df['payment'] == 'credit card'].copy()
card['no_tip'] = card['tip'] == 0
tip_rate = lambda s: (s > 0).mean() * 100
number_of_rides = card.pivot_table(index='pickup_borough', columns='dropoff_borough', values='tip', aggfunc='count', margins=True)
tip_rate_by_area = card.pivot_table(index='pickup_borough', columns='dropoff_borough', values='tip', aggfunc=tip_rate, margins=True)
tip_rate_by_area.where(number_of_rides > 30).round(1)

dropoff_borough,Bronx,Brooklyn,Manhattan,Queens,Staten Island,All
pickup_borough,,,,,,
Bronx,8.9,NaN,NaN,NaN,NaN,6.8
Brooklyn,NaN,48.3,64.3,NaN,NaN,49.2
Manhattan,NaN,88.8,96.4,93.0,NaN,95.7
Queens,NaN,75.0,92.0,65.8,NaN,78.4
All,21.4,65.1,95.2,73.7,NaN,90.1


Overall 90% of card payments include a tip. The tip rate strongly varies by pickup borough: 96% for Manhattan pickups, 78% for Queens, 49% for Brooklyn and only 7% for Bronx. The destination also matters, 95% of rides ending in Manhattan includes a tip, compared with 21% for ending in Bronx. Riders from Queens and Brooklyn tip more often when they go to Manhattan (64% and 92%), compared to when they stay in their own borough (48% and 66%).

Cash riders were excluded because cash tips are not recorded, as were 44 rides with no payment type. Routes with 30 or less rides are hidden as unreliable.

In [35]:
tips_by_color = card.groupby('color').agg(
    rides=('tip', 'count'),
    tip_rate=('tip', tip_rate)
).round(1)
display(tips_by_color)
color_by_area = card.pivot_table(index='pickup_borough', columns='color', values='tip', aggfunc='count')
display(color_by_area)
co_ar_tip = card.pivot_table(index='pickup_borough', columns='color', values='tip', aggfunc=tip_rate)
co_ar_tip.where(color_by_area > 30).round(1)

,rides,tip_rate
color,,
green,577,53.2
yellow,4000,95.4


color,green,yellow
pickup_borough,,
Bronx,62,12
Brooklyn,216,45
Manhattan,155,3684
Queens,141,242


color,green,yellow
pickup_borough,,
Bronx,8.1,NaN
Brooklyn,46.3,62.2
Manhattan,80.6,96.3
Queens,54.6,92.6


Taxi color vs borough. Yellow taxis have a much higher card tip rate than green ones (95% vs 53%), and green taxis dominate the Bronx and Brooklyn. However, the borough difference remains within each color: green taxis reach 81% tip rate in Manhattan but only 8% in Bronx. Both cab (taxi) type and location are factors in how often riders tip.